In [ ]:
!pip install xlrd

In [ ]:
import pandas as pd


TIN_PATH = "/content/drive/MyDrive/drone iit/Tin_feature_matrix.csv"
RCC_PATH = "/content/drive/MyDrive/drone iit/RCC_Building_feature_matrix.csv"

OUTPUT_PATH = "/content/drive/MyDrive/drone iit/Tin_vs_RCC_final.csv"

tin_df = pd.read_csv(TIN_PATH)
rcc_df = pd.read_csv(RCC_PATH)

tin_df.columns = tin_df.columns.str.strip()
rcc_df.columns = rcc_df.columns.str.strip()

tin_df = tin_df[tin_df['Tin_class'] == 1].reset_index(drop=True)
rcc_df = rcc_df[rcc_df['RCC_Building_class'] == 1].reset_index(drop=True)

def clean_columns(df):
    new_cols = []
    for col in df.columns:
        if "Tin_" in col:
            col = col.replace("Tin_", "")
        if "RCC_" in col:
            col = col.replace("RCC_", "")
        if "Building_" in col:
            col = col.replace("Building_", "")
        new_cols.append(col)
    df.columns = new_cols
    return df

tin_df = clean_columns(tin_df)
rcc_df = clean_columns(rcc_df)

tin_df['target'] = 1
rcc_df['target'] = 0

tin_df = tin_df.drop(columns=['class'], errors='ignore')
rcc_df = rcc_df.drop(columns=['class'], errors='ignore')

df = pd.concat([tin_df, rcc_df], ignore_index=True)

df = df.sort_values(by='target', ascending=False)

df = df.drop_duplicates(subset=['Tile_Name'])

df = df.fillna(0)

df.to_csv("TIN_VS_RCC.csv", index=False)

print("✅ Final dataset ready!")
print("Shape:", df.shape)
print(df.head())

✅ Final dataset ready!
Shape: (1080, 32)
      Tile_Name    R_mean    R_std  R_min  R_max    G_mean    G_std  G_min  \
735  image_1335  216.6195  29.9012   14.0  255.0  233.5799  30.6800   16.0   
734  image_1329  108.4159  40.1772    4.0  255.0  114.0746  41.8752    2.0   
733  image_1328  127.5277  36.8164    2.0  255.0  131.1817  39.7215    8.0   
732  image_1327  168.7281  31.0381   20.0  252.0  171.1648  33.4774   25.0   
731  image_1326  106.1366  38.5388   12.0  200.0  107.5740  36.5754   12.0   

     G_max    B_mean  ...  Texture_correlation    Area  Perimeter  \
735  255.0  239.8944  ...               0.9743  3966.0    62.9762   
734  255.0  128.5444  ...               0.9747  4677.0    68.3886   
733  255.0  142.7904  ...               0.9896  6793.0    82.4197   
732  255.0  175.7151  ...               0.9934  4230.0    65.0385   
731  204.0   99.8569  ...               0.9867  1230.0    35.0714   

     Compactness  Hist_R  Hist_G  Hist_B  Saturation_mean  Brightness_mean 

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

df = pd.read_csv("/content/drive/MyDrive/drone iit/TIN_VS_RCC.csv")

X = df.drop(columns=['target', 'Tile_Name'], errors='ignore')
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#Random Forest model
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_probs = rf.predict_proba(X_test)[:, 1]

#MLP model
mlp = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

mlp.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

mlp.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    verbose=0
)

mlp_probs = mlp.predict(X_test_scaled).flatten()

final_probs = (rf_probs + mlp_probs) / 2

y_pred = (final_probs > 0.5).astype(int)

print("\n✅ ENSEMBLE ACCURACY:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


joblib.dump(rf, "/content/drive/MyDrive/drone iit/rf_model.pkl")
joblib.dump(scaler, "/content/drive/MyDrive/drone iit/scaler.pkl")
mlp.save("/content/drive/MyDrive/drone iit/mlp_model.h5")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 

✅ ENSEMBLE ACCURACY: 0.8935185185185185

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.71      0.81        69
           1       0.88      0.98      0.93       147

    accuracy                           0.89       216
   macro avg       0.91      0.84      0.87       216
weighted avg       0.90      0.89      0.89       216

